# Notebook 01a: Rubric Iteration
This notebook implements the iterative development of the severity rubric. We derive severity levels (0-3) based on physical outcomes reported in the incident.

In [ ]:
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

with open('configs/main_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

with open('configs/severity_rubric_v1.yaml', 'r') as f:
    rubric_spec = yaml.safe_load(f)

df = pd.read_parquet(config['paths']['raw_data'])
print(f"Loaded {len(df)} records.")

## 1. Define Rubric Logic (v1)

In [ ]:
def calculate_severity(row):
    result = str(row.get('Events.5_Result', '')).lower()
    injury_cols = [c for c in row.index if 'Injury' in c]
    injuries = [str(row.get(c, '')).lower() for c in injury_cols]
    miss_dist = str(row.get('Events.1_Miss Distance', '')).lower().strip()
    
    # Level 3 — Critical
    if ('destroyed' in result 
        or any('fatal' in inj for inj in injuries)
        or 'evacuation' in result
        or 'loss of control' in result):
        return 3
    
    # Level 2 — Substantial
    if ('substantial damage' in result
        or any(inj in {'minor', 'serious'} for inj in injuries)
        or 'emergency' in result):
        return 2
    
    # Level 1 — Moderate
    close_miss_categories = {'less than 100 ft', '100-200 ft', '200-500 ft'}
    if ('minor damage' in result 
        or miss_dist in close_miss_categories):
        return 1
        
    return 0

df['severity_level'] = df.apply(calculate_severity, axis=1)
print("Severity calculation complete.")

## 2. Distribution Validation

In [ ]:
dist = df['severity_level'].value_counts(normalize=True).sort_index() * 100
print("Severity Distribution (%):")
print(dist)

plt.figure(figsize=(10, 6))
sns.barplot(x=dist.index, y=dist.values)
plt.title('Severity Level Distribution')
plt.ylabel('Percentage')
plt.xlabel('Severity Level (0=Minor, 3=Critical)')
plt.show()

## 3. Face Validity Check (Sampling)
Reviewing random samples from each level to ensure they match the labels.

In [ ]:
for level in range(4):
    print(f"\n--- SAMPLES FOR LEVEL {level} ---")
    samples = df[df['severity_level'] == level].sample(min(3, len(df[df['severity_level'] == level])))
    for i, row in samples.iterrows():
        print(f"ACN: {row.get('acn_num_ACN')}")
        print(f"Result: {row.get('Events.5_Result')}")
        print(f"Miss Dist: {row.get('Events.1_Miss Distance')}")
        print(f"Synopsis: {str(row.get('Report 1.2_Synopsis'))[:200]}...")
        print("-" * 30)

## 4. Save Targets

In [ ]:
df[['acn_num_ACN', 'severity_level']].to_parquet('data/processed/severity_targets.parquet')
print("Targets saved.")